In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import sequence

In [3]:
## Load dataset indexes
imdb_index = imdb.get_word_index()
sentence_dict = {value: key for key, value in imdb_index.items()}

In [4]:
## Loading model
model = load_model('./rnn_imdb_ac9901.h5')
model.summary()

I0000 00:00:1768266889.815519   16048 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10231 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:29:00.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 64)               │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (32, 1)                │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,321,219 (5.04 MB)

 Trainable params: 1,321,217 (5.04 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [ ]:
##Checking weights
model.get_weights()

[array([[-5.24424076e-01, -4.79841948e-01,  6.52543843e-01, ...,
          7.76045978e-01, -9.49775934e-01,  6.90517604e-01],
        [-4.94717471e-02, -1.64981820e-02,  1.06591590e-01, ...,
         -1.31515376e-02, -1.26647651e-01,  8.87938365e-02],
        [ 4.79045101e-02,  1.93788230e-01, -5.50302006e-02, ...,
          1.00421816e-01, -1.28659606e-01, -1.15770936e-01],
        ...,
        [-8.43443722e-02,  6.21780287e-04, -7.64583945e-02, ...,
         -1.84808262e-02,  3.69086787e-02,  2.22921539e-02],
        [-3.32118049e-02, -1.82543378e-02, -9.85404756e-03, ...,
          5.14501631e-02, -3.52491774e-02, -2.29285508e-02],
        [-8.80791694e-02, -1.41152203e-01, -5.02228774e-02, ...,
          6.71234578e-02,  3.71020623e-02,  1.50647789e-01]],
       shape=(10000, 128), dtype=float32),
 array([[-0.14928634, -0.06989743,  0.14628163, ..., -0.05755524,
         -0.12584296,  0.11778019],
        [-0.11300116,  0.02315726, -0.00319697, ..., -0.03990266,
          0.0088633

In [10]:
## Function to preprocess user input
def preprocess_user_input(review, word_index=imdb_index):
    words = review.lower().split()
    review_vec = [word_index.get(word,2)+3 for word in words]
    padded_review = sequence.pad_sequences([review_vec],maxlen=500)
    return padded_review

## Function to predict the user input
def predict_review(review):
    preprocessed_input = preprocess_user_input(review)

    prediction = model.predict(preprocessed_input)

    review_sentiment = 'Negetive' if prediction[0][0] < 0.5 else 'Positive'

    return prediction[0][0], review_sentiment

In [23]:
## Example reviews
test_review = [
    "I loved this movie, the acting was superb!",
    "I've seen some bad movies in my time, but this one takes the cake. The writing was atrocious, the acting was laughable, and the plot made no sense whatsoever. It felt like the filmmakers threw a bunch of random elements into a blender and hit puree. The result was a hot mess that left me checking my watch multiple times. Don't bother with this one unless you want to punish yourself.",
    "I was really looking forward to this movie, the plot was interesting and the characters were well-developed, but the ending was a bit disappointing.",
    "What could've been a great movie ended up being a complete disaster. The concept was interesting, but the execution was terrible. The characters were cardboard cutouts, the dialogue was cramped, and the ending was unsatisfying. I left the theater feeling ripped off, like I'd paid money to watch something that wasn't finished. Unless you're a glutton for punishment, stay away from this one.",
    "This movie was a bit of a rollercoaster for me. I loved the unique concept and the way it explored certain themes, but it felt like it lost its way in the second half. The characters were interesting, but some of their arcs felt incomplete or predictable. The visuals were stunning, though, and the performances were solid. If you're a fan of the genre, you might enjoy it, but others might find it a bit hit-or-miss. I'd say give it a watch and see what you think."
]

for review in test_review:
    score, sentiment = predict_review(review)
    print(f"User Review: {review}\n")
    print(f"Score: {score}\n")
    print(f"Sentiment: {sentiment}")
    print("-"*10,"\n\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
User Review: I loved this movie, the acting was superb!

Score: 0.7461203932762146

Sentiment: Positive
---------- 


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
User Review: I've seen some bad movies in my time, but this one takes the cake. The writing was atrocious, the acting was laughable, and the plot made no sense whatsoever. It felt like the filmmakers threw a bunch of random elements into a blender and hit puree. The result was a hot mess that left me checking my watch multiple times. Don't bother with this one unless you want to punish yourself.

Score: 0.1612304151058197

Sentiment: Negetive
---------- 


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
User Review: I was really looking forward to this movie, the plot was interesting and the characters were well-developed, but the ending was a bit disappointing.

Score: 0.32109692692756653

Sentiment: Negetive
---------- 


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
User Review: What could've been a great movi